# Analys av GPS-spår (Uppsala)

I det här projektet analyserar jag mina egna GPS-spår från vandringar i skogen runt Uppsala.

## 💎 Syfte

Målet är att:

- Visualisera rörelsemönster
- Beräkna hastighet från rå GPS-data
- Klassificera rörelse (walking vs car)
- Filtrera bort brus i datan

---

## 💎 Data

Jag använder en app som heter "FootMark" på min iPhone 12. Varje gång jag tar en promenad i skogen startar jag appen och spelar in min vandringsdata. Sedan kan jag enkelt importera datan till datorn. Förresten kostar appen bara 39 kr per år. Datan består av .gpx-filer, där varje fil representerar en inspelad rutt.

- latitud (latitude)
- longitud (longitude)
- höjd (elevation)
- tidsstämpel (time)

---

## 💙 Steg 1: Läs in och strukturera data

Vi läser in alla GPX-filer och konverterar dem till en strukturerad tabell.

👉 Viktigt: Punkten måste behålla rätt ordning (tidssekvens), annars blir beräkningarna fel.

In [33]:
#| code-fold: true
from pathlib import Path
import gpxpy
import pandas as pd

gpx_folder = Path("dataCollection/geoData/")

rows = []

for gpx_file in gpx_folder.glob("*.gpx"):
    with open(gpx_file, "r", encoding="utf-8") as f:
        gpx = gpxpy.parse(f)

    for track_id, track in enumerate(gpx.tracks):
        for segment_id, segment in enumerate(track.segments):
            for point_id, p in enumerate(segment.points):
                rows.append({
                    "file": gpx_file.stem,
                    "track_id": track_id,
                    "segment_id": segment_id,
                    "point_id": point_id,
                    "latitude": p.latitude,
                    "longitude": p.longitude,
                    "elevation": p.elevation,
                    "time": p.time,
                })

df = pd.DataFrame(rows)
df["file_time"] = pd.to_datetime(df["file"], format="%Y.%m.%d-%H.%M.%S")
df = df.sort_values(["file_time", "point_id"])
print("All files:",df["file"].unique())
df = df.drop(columns=["track_id", "segment_id"])
print("Examples from the data: \n",df.head())

All files: <StringArray>
['2026.03.01-13.04.11', '2026.03.02-16.06.09', '2026.03.06-14.44.02',
 '2026.03.08-14.54.04', '2026.03.15-13.53.42', '2026.03.24-10.38.20',
 '2026.03.27-16.39.50', '2026.04.12-16.10.29', '2026.04.16-12.32.37',
 '2026.04.18-17.16.01', '2026.05.02-18.49.47']
Length: 11, dtype: str
Examples from the data: 
                     file  point_id   latitude  longitude  elevation  \
198  2026.03.01-13.04.11         0  59.793252  17.684703  64.459105   
199  2026.03.01-13.04.11         1  59.793142  17.685584  63.132652   
200  2026.03.01-13.04.11         2  59.791022  17.689015  64.826286   
201  2026.03.01-13.04.11         3  59.790575  17.689117  65.318291   
202  2026.03.01-13.04.11         4  59.790366  17.688319  64.272516   

                         time           file_time  
198 2026-03-01 12:04:11+00:00 2026-03-01 13:04:11  
199 2026-03-01 12:06:08+00:00 2026-03-01 13:04:11  
200 2026-03-01 12:20:28+00:00 2026-03-01 13:04:11  
201 2026-03-01 12:22:35+00:00 2026


## 💙  Steg 2: Visualisera rörelsemönster

### 2.1 Skapa baskarta

Vi använder `folium` för att skapa en interaktiv karta centrerad kring Uppsala.

In [35]:
import folium
m_uppsala = folium.Map(
    location=[59.8586, 17.6389],  # Uppsala center
    zoom_start=12,
    tiles="OpenStreetMap"
)

### 2.2 Lägg till GPS-spår

Varje rutt ritas som en linje på kartan.

👉 Viktigt:
- Punkterna måste sorteras korrekt
- Annars ritas rutten fel  

In [36]:
all_points = []

for file_name, group in df.groupby("file"):
    # 🔑 VERY IMPORTANT: keep correct order
    group = group.sort_values("point_id")

    points = group[["latitude", "longitude"]].values.tolist()

    if len(points) > 1:
        folium.PolyLine(
            points,
            weight=3,
            opacity=0.8,
            tooltip=file_name
        ).add_to(m_uppsala)

        all_points.extend(points)

# auto zoom
if all_points:
    m_uppsala.fit_bounds(all_points)

m_uppsala

### ⚠ OBS!

***Jag glömde stänga av appen när jag körde!***

## 💙 Steg 3: Klassificera rörelse

### Idé: Separera data baserat på hastighet

Grundtanken är att olika typer av rörelse har olika typiska hastigheter.

Till exempel:

- Walking → låg hastighet
- Car → högre hastighet

Genom att beräkna hastigheten mellan varje GPS-punkt kan vi därför använda hastighet som en enkel indikator för att skilja mellan olika rörelsetyper.

---

### Steg 3.1: Beräkna hastighet från rådata

För små områden (som Uppsala) kan vi använda en enkel approximation:

- 1 grad latitud ≈ 111 320 meter
- 1 grad longitud ≈ 111 320 × cos(latitud)

#### Avstånd mellan två punkter

distance = sqrt(dx^2 + dy^2)

Där:

- dx = (longitude2 - longitude1) × meter per longitud
- dy = (latitude2 - latitude1) × meter per latitud

#### Hastighet

speed = distance / tid

(omvandlas till km/h genom att multiplicera med 3.6)

---

#### Implementering

Vi beräknar:

- Avstånd mellan punkter
- Tidsdifferens
- Hastighet (km/h)

In [37]:
from math import sqrt, cos, radians

from math import cos, radians

UPPSALA_LAT = 59.8586

LAT_M = 111_320
LON_M = 111_320 * cos(radians(UPPSALA_LAT))

df_speed = df.copy()

# previous values
df_speed["lat_prev"] = df_speed["latitude"].shift(1)
df_speed["lon_prev"] = df_speed["longitude"].shift(1)
df_speed["time_prev"] = df_speed["time"].shift(1)
df_speed["file_prev"] = df_speed["file"].shift(1)

# same file check
same_file = df_speed["file"] == df_speed["file_prev"]

# compute dx, dy directly
df_speed["dy"] = (df_speed["latitude"] - df_speed["lat_prev"]) * LAT_M
df_speed["dx"] = (df_speed["longitude"] - df_speed["lon_prev"]) * LON_M

# distance
df_speed["distance_m"] = (df_speed["dx"]**2 + df_speed["dy"]**2) ** 0.5
df_speed.loc[~same_file, "distance_m"] = None

# Time difference
df_speed["time_diff"] = (
    (df_speed["time"] - df_speed["time_prev"])
    .dt.total_seconds()
)

df_speed.loc[~same_file, "time_diff"] = None

# Speed
df_speed["speed_kmh"] = (df_speed["distance_m"] / df_speed["time_diff"]) * 3.6

### Steg 3.2: Klassificera rörelse utifrån hastighet

Vi filtrerar också bort:

- negativa tidssteg
- orimliga hastigheter (t.ex. > 150 km/h)
-
Vi klassificerar rörelse baserat på hastighet:

- ≤ 9 km/h → walking
- \> 9 km/h → car


In [38]:
# Car vs Waling
MAX_WALKING_SPEED_KMH = 9
# clean bad values
df_speed.loc[df_speed["time_diff"] <= 0, "speed_kmh"] = None
df_speed.loc[df_speed["speed_kmh"] > 150, "speed_kmh"] = None

df_speed["mode"] = "walking"
df_speed.loc[df_speed["speed_kmh"] > MAX_WALKING_SPEED_KMH, "mode"] = "car"
df_speed.loc[df_speed["speed_kmh"].isna(), "mode"] = None

print("Dataframe Reveal: \n", df_speed[902:906])

Dataframe Reveal: 
                     file  point_id   latitude  longitude  elevation  \
951  2026.03.24-10.38.20       249  59.870134  17.655526  12.621190   
952  2026.03.24-10.38.20       250  59.869967  17.655897  12.614025   
953  2026.03.24-10.38.20       251  59.869791  17.656285  12.568982   
954  2026.03.24-10.38.20       252  59.869611  17.656693  12.220599   

                         time           file_time   lat_prev   lon_prev  \
951 2026-03-24 10:21:21+00:00 2026-03-24 10:38:20  59.870304  17.655148   
952 2026-03-24 10:21:23+00:00 2026-03-24 10:38:20  59.870134  17.655526   
953 2026-03-24 10:21:25+00:00 2026-03-24 10:38:20  59.869967  17.655897   
954 2026-03-24 10:21:27+00:00 2026-03-24 10:38:20  59.869791  17.656285   

                    time_prev            file_prev         dy         dx  \
951 2026-03-24 10:21:19+00:00  2026.03.24-10.38.20 -18.934276  21.131373   
952 2026-03-24 10:21:21+00:00  2026.03.24-10.38.20 -18.602824  20.755945   
953 2026-03-24 10:21

### Steg 3.3: Visualisering:

- 🔵 blå = walking
- 🟠 orange = car

In [39]:
import folium

m_speed = folium.Map(
    location=[59.8586, 17.6389],
    zoom_start=12,
    tiles="OpenStreetMap"
)

for file_name, group in df_speed.groupby(["file", "mode"]):
    mode = group["mode"].iloc[0]
    points = group[["latitude", "longitude"]].values.tolist()

    color = "blue" if mode == "walking" else "orange"

    for p in points:
        folium.CircleMarker(
            location=p,
            radius=1,
            color=color,
            fill=True,
            tooltip=f"{file_name} - {mode}"
        ).add_to(m_speed)

m_speed

### ⚠ OBS!
***Felklassificering vid stopp***

Ett problem uppstår när bilen står stilla, till exempel vid ett rödljus.

Då blir hastigheten nära 0 → klassificeras som "walking", vilket är fel.

---

## 💙  Steg 4: Smoothing

Vi förbättrar klassificeringen genom att titta på närliggande punkter.

###  Idé:

Om en punkt är "walking", men:

- majoriteten av närliggande punkter är "car"

→ då bör den klassificeras som "car"

---
###  Parametrar

- WINDOW = 5  (antal punkter före och efter)
- THRESHOLD = 0.5  (andel bil-punkter)

---

In [40]:
# Calculate the points near the walking points
WINDOW = 5
THRESHOLD = 0.5

all_files = df_speed["file"].unique()

# create new column
df_speed["mode_smooth"] = df_speed["mode"]

for file_name in all_files:

    # get one file
    mask = df_speed["file"] == file_name
    sub = df_speed.loc[mask].sort_values("point_id")

    modes = sub["mode"].tolist()
    new_modes = modes.copy()

    for i, mode in enumerate(modes):

        # only adjust walking points
        if mode != "walking":
            continue

        # define window
        start = max(0, i - WINDOW)
        end = min(len(modes), i + WINDOW + 1)

        # exclude itself
        nearby = modes[start:i] + modes[i+1:end]

        if len(nearby) == 0:
            continue

        car_ratio = nearby.count("car") / len(nearby)

        # if more than 50% are car → change to car
        if car_ratio > THRESHOLD:
            new_modes[i] = "car"

    # write back to dataframe
    df_speed.loc[sub.index, "mode_smooth"] = new_modes

In [42]:
print("With the mode_smooth column: \n",df_speed.head())

With the mode_smooth column: 
                     file  point_id   latitude  longitude  elevation  \
198  2026.03.01-13.04.11         0  59.793252  17.684703  64.459105   
199  2026.03.01-13.04.11         1  59.793142  17.685584  63.132652   
200  2026.03.01-13.04.11         2  59.791022  17.689015  64.826286   
201  2026.03.01-13.04.11         3  59.790575  17.689117  65.318291   
202  2026.03.01-13.04.11         4  59.790366  17.688319  64.272516   

                         time           file_time   lat_prev   lon_prev  \
198 2026-03-01 12:04:11+00:00 2026-03-01 13:04:11        NaN        NaN   
199 2026-03-01 12:06:08+00:00 2026-03-01 13:04:11  59.793252  17.684703   
200 2026-03-01 12:20:28+00:00 2026-03-01 13:04:11  59.793142  17.685584   
201 2026-03-01 12:22:35+00:00 2026-03-01 13:04:11  59.791022  17.689015   
202 2026-03-01 12:26:25+00:00 2026-03-01 13:04:11  59.790575  17.689117   

                    time_prev            file_prev          dy          dx  \
198          

In [43]:
print("Before smoothing: \n", df_speed["mode"].value_counts(dropna=False))

Before smoothing: 
 mode
walking    1459
car         459
NaN          11
Name: count, dtype: int64


In [44]:
print("After smoothing: \n", df_speed["mode_smooth"].value_counts(dropna=False))

After smoothing: 
 mode_smooth
walking    1415
car         503
NaN          11
Name: count, dtype: int64


## 💙 Resultat

Efter smoothing får vi en mer stabil klassificering:

- Brus reduceras
- Kortvariga stopp påverkar inte resultatet lika mycket

Visualisering:

- 🔵 blå = walking
- 🟠 orange = car

---

In [45]:
import folium

df_plot = df_speed.copy()

m_smooth = folium.Map(
    location=[59.8586, 17.6389],
    zoom_start=12,
    tiles="OpenStreetMap"
)

all_points = []

for file_name, group in df_plot.groupby(["file", "mode_smooth"]):
    mode = group["mode_smooth"].iloc[0]
    points = group[["latitude", "longitude"]].values.tolist()

    color = "blue" if mode == "walking" else "orange"

    folium.PolyLine(
        points,
        color=color,
        weight=4,
        opacity=0.8,
        tooltip=f"{file_name} - {mode}"
    ).add_to(m_smooth)

    all_points.extend(points)

m_smooth.fit_bounds(all_points)

m_smooth

Nu är jag glad!